## Use of Sqlitesearch

In [10]:
from sqlitesearch import TextSearchIndex

sqlite_index = TextSearchIndex(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"],
    db_path="faq.db"
)

In [18]:
sqlite_index.search('how do I join the course ?')

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'question': 'Overview',
  'section': 'GitHub README',
  'answer': '<div align="center">\n\n<img width="80%" src="images/llm-zoomcamp-2026.jpg" alt="LLM Zoomcamp 2026 - Free Course on Building LLM Applications with RAG, Agents, and Vector Search" />\n\n<h1>LLM Zoomcamp: Free Course on Building LLM Applications with RAG, Agents & Vector Search</h1>\n<h3>Go from LLM basics to a production-ready AI 

## Use of Elasticsearch

In [1]:
from elasticsearch import Elasticsearch

es_client = Elasticsearch("http://localhost:9200")

index_name = "faq-llm-zoomcamp"

# Separation of settings and mappings for the Elasticsearch index
settings = {"number_of_shards": 1, "number_of_replicas": 0}

mappings = {
    "properties": {
        "question": {"type": "text"},
        "section": {"type": "text"},
        "answer": {"type": "text"},
        "course": {"type": "keyword"},
    }
}

# Deletion of the old index if it exists
try:
    es_client.indices.delete(index=index_name)
    print(f"Old index '{index_name}' deleted.")
except Exception:
    pass

# Creation of the index with explicit settings and mappings
es_client.indices.create(
    index=index_name,
    settings=settings,
    mappings=mappings
)
print(f"Index '{index_name}' created with success!")

Old index 'faq-llm-zoomcamp' deleted.
Index 'faq-llm-zoomcamp' created with success!


In [2]:
from rag_helper  import ElasticsearchIndex

es_index = ElasticsearchIndex(
    es_client=es_client, index_name="faq-llm-zoomcamp"
)

In [3]:
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
openai_client = OpenAI()

In [11]:
from rag_helper import RAGBase

assistant = RAGBase(index=sqlite_index, llm_client=openai_client)
#assistant = RAGBase(index=es_index, llm_client=openai_client)

In [12]:
answer = assistant.rag('I just discovered the course, can I still join ?')
print(answer)

Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


In [13]:
answer = assistant.rag('Is this course suitable compared to the job market ?')
print(answer)

I don't know.


In [14]:
answer = assistant.rag('Is this course suitable for a data engineer ?')
print(answer)

I don't know.


In [ ]:
custom_instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=sqlite_index,
    llm_client=openai_client,
    instructions=custom_instructions,
)

In [11]:
answer = assistant.rag('I just discovered the course, can I still join ?')
print(answer)

Yes — you can still join. If you want to receive a certificate, make sure to submit your project while submissions are still being accepted.


In [12]:
assistant.rag("How do I get a certificate?")

'To get a certificate, you need to finish the course with a **live cohort**. You must also:\n\n- **Pass the capstone project**\n- **Complete the required peer reviews**\n\nHomework is **not required** for the certificate. You can prepare in self-paced mode, but **project submission and peer review must happen while a live cohort is accepting them**.'

In [13]:
sqlite_index.close()